# Generativa — Pruebas End-to-End en GPU + Túnel Remoto (Google Colab)

Este notebook ejecuta el **Worker de Imágenes** de `GeneradorIA` sobre una **GPU NVIDIA (CUDA `float16`)** en Google Colab para:
1. **Validar de punta a punta todos los endpoints pendientes** que en CPU tardaban 15–30 minutos o crasheaban en la iGPU Intel:
   - `/generate` (txt2img + `hires_fix` + restauración facial `GFPGAN` + escalado Full HD `FSRCNN`)
   - `/edit` (img2img)
   - `/inpaint` (edición localizada con segmentación automática `SegFormer`: `ropa`, `fondo`, `persona`, `rostro` + `hires_fix`)
   - `/generate-controlled` (`ControlNet` con `pose` vía OpenPose y `edges` vía Canny)
   - `/generate-with-reference` (`IP-Adapter` `ip-adapter-full-face_sd15.bin` para mantener identidad en escena nueva)
   - `/faceswap` (`InsightFace buffalo_l` + `inswapper_128.onnx` + `GFPGANv1.4.pth`)
   - `/upscale` (`OpenCV dnn_superres` `FSRCNN_x4.pb` a 1920px Full HD)
2. **Exponer el worker vía túnel HTTPS gratuito (`cloudflared`)** para conectarlo a tu Frontend Next.js y API C# locales usando `start-dev-colab.ps1`.

> **Requisito antes de empezar:** En el menú superior de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**.

In [ ]:
# 1. Verificar GPU NVIDIA y clonar/actualizar el repositorio
import os
import subprocess
import torch

print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detectada  : {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("⚠️ No se detectó GPU CUDA. Ve a 'Entorno de ejecución -> Cambiar tipo de entorno de ejecución' y elige 'T4 GPU'.")

REPO_DIR = "/content/GeneradorIA"
REPO_URL = "https://github.com/jfrodriguezva/GeneradorIA.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Directorio activo: {os.getcwd()}")

In [ ]:
# 2. Instalar dependencias del Worker de Imágenes para Colab (GPU CUDA)
# Nota: reinstalamos opencv-contrib-python-headless al final porque controlnet_aux
# arrastra opencv-python-headless, y dnn_superres (upscale) solo viene en 'contrib'.
!pip install -q -r worker-python/image/requirements-colab.txt
!pip install -q --force-reinstall --no-deps opencv-contrib-python-headless
print("✅ Dependencias instaladas correctamente.")

In [ ]:
# 3. Descargar automáticamente los modelos auxiliares (FSRCNN_x4.pb, GFPGANv1.4.pth, inswapper_128.onnx)
!python worker-python/image/download_models.py

In [ ]:
# 4. Arrancar el Worker de Imágenes en segundo plano con GENERATIVA_IMAGE_DEVICE=CUDA
import os
import subprocess
import time
import urllib.request
import json

# Matar instancia previa si se re-ejecuta la celda
subprocess.run(["pkill", "-f", "worker-python/image/main.py"], check=False)

env = os.environ.copy()
env["GENERATIVA_IMAGE_DEVICE"] = "CUDA"
env["PYTHONUNBUFFERED"] = "1"

log_file = open("/content/image_worker.log", "w")
worker_proc = subprocess.Popen(
    ["python", "worker-python/image/main.py"],
    cwd="/content/GeneradorIA",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("⏳ Esperando a que el Worker de Imágenes descargue/cargue epiCRealism en GPU (puede tardar 1-2 min la primera vez)...")
ready = False
for i in range(180):
    if worker_proc.poll() is not None:
        print("❌ El proceso del worker terminó inesperadamente. Últimas líneas del log:")
        with open("/content/image_worker.log") as f:
            print(f.read()[-3000:])
        raise RuntimeError("Worker falló al iniciar")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8002/health", timeout=3) as resp:
            data = json.loads(resp.read().decode())
            if data.get("model_loaded"):
                print(f"✅ Worker listo en GPU: {data}")
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    with open("/content/image_worker.log") as f:
        print(f.read()[-3000:])
    raise TimeoutError("El worker no respondió a tiempo.")

In [ ]:
# 5. Ejecutar todas las pruebas End-to-End de los endpoints pendientes
import sys
sys.path.insert(0, "/content/GeneradorIA/worker-python/image")
from test_e2e_endpoints import run_suite

OUT_DIR = "/content/GeneradorIA/test_outputs"
results = run_suite(base_url="http://127.0.0.1:8002", out_dir=OUT_DIR, steps=25)

In [ ]:
# 6. Visualizar la galería de resultados de todas las pruebas E2E
import glob
import os
import matplotlib.pyplot as plt
from PIL import Image

image_files = sorted(glob.glob("/content/GeneradorIA/test_outputs/*.png"))
cols = 3
rows = (len(image_files) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes = axes.flatten() if len(image_files) > 1 else [axes]

for ax, img_path in zip(axes, image_files):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(f"{os.path.basename(img_path)}\n({img.width}x{img.height}px)", fontsize=11)
    ax.axis("off")

for ax in axes[len(image_files):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Conectar este Worker GPU de Colab con tu App Local (Frontend Next.js + API C#)

Ejecuta la siguiente celda para abrir un **túnel público HTTPS gratuito** con Cloudflare (`cloudflared`, no requiere cuenta ni token).

Cuando aparezca la URL `https://xxxx.trycloudflare.com`, copia el comando que te muestra la celda y ejecútalo en PowerShell en tu PC local:
```powershell
powershell -File start-dev-colab.ps1 -ColabUrl "https://xxxx.trycloudflare.com"
```

In [ ]:
# 7. Abrir túnel HTTPS con Cloudflared hacia el puerto 8002
import os
import re
import subprocess
import time

if not os.path.isfile("/usr/local/bin/cloudflared"):
    print("Descargando cloudflared...")
    subprocess.run([
        "wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"
    ], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

subprocess.run(["pkill", "-f", "cloudflared"], check=False)
tunnel_log = open("/content/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8002"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    with open("/content/cloudflared.log") as f:
        content = f.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if public_url:
    print("=" * 70)
    print(f"🌐 TÚNEL ACTIVO: {public_url}")
    print(f"🩺 Healthcheck : {public_url}/health")
    print("=" * 70)
    print("\nEjecuta esto en tu PC local (PowerShell en la raíz del proyecto):\n")
    print(f'powershell -File start-dev-colab.ps1 -ColabUrl "{public_url}"')
    print("\n(Deja esta sesión de Colab abierta mientras uses la aplicación local)")
else:
    with open("/content/cloudflared.log") as f:
        print(f.read())
    raise RuntimeError("No se pudo obtener la URL de Cloudflared.")